In [3]:
# ============================================================
# CELL 1 — load_raw_data.py
# Loads all 13 CSVs into a dict of DataFrames  →  dfs
# ============================================================

import os
import pandas as pd
import numpy as np

# ── Set your raw data folder here ────────────────────────────
DATA_PATH = 'D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/data/raw'
# ─────────────────────────────────────────────────────────────

SOURCE_MAP = {
    "customer":       "CUSTOMER_MASTER.csv",
    "loan":           "LOAN_MASTER_DAILY.csv",
    "application":    "APPLICATION_FORM_DATA.csv",
    "bureau":         "CIBIL_RAW_PULL.csv",
    "bureau_refresh": "BUREAU_REFRESH_6M.csv",
    "bank_statement": "BANK_STATEMENT_SUMMARY.csv",
    "income":         "INCOME_DOCUMENT_PARSED.csv",
    "dpd_history":    "DPD_HISTORY_MONTHLY.csv",
    "portfolio":      "ACCOUNT_PORTFOLIO_MONTHLY.csv",
    "repayment":      "REPAYMENT_TRANSACTIONS.csv",
    "settlement":     "SETTLEMENT_CASES.csv",
    "writeoff":       "WRITE_OFF_DATA.csv",
    "disbursement":   "DISBURSEMENT_DETAILS.csv",
}

dfs = {}
load_summary = []

print("=" * 60)
print("  STEP 1 — LOAD RAW DATA")
print("=" * 60)

for key, fname in SOURCE_MAP.items():
    fpath = os.path.join(DATA_PATH, fname)
    if os.path.exists(fpath):
        dfs[key] = pd.read_csv(fpath, low_memory=False)
        status = "OK"
        rows   = dfs[key].shape[0]
        cols   = dfs[key].shape[1]
        nulls  = round(dfs[key].isnull().mean().mean() * 100, 2)
    else:
        status = "NOT FOUND"
        rows, cols, nulls = 0, 0, 0.0

    load_summary.append({
        "table":    key,
        "file":     fname,
        "status":   status,
        "rows":     rows,
        "columns":  cols,
        "nulls_%":  nulls,
    })

load_summary_df = pd.DataFrame(load_summary)

print(load_summary_df.to_string(index=False))
print(f"\n  Loaded : {len(dfs)}/{len(SOURCE_MAP)} tables")
print("=" * 60)


  STEP 1 — LOAD RAW DATA
         table                          file status  rows  columns  nulls_%
      customer           CUSTOMER_MASTER.csv     OK   400       19     0.00
          loan         LOAN_MASTER_DAILY.csv     OK   500       17     0.00
   application     APPLICATION_FORM_DATA.csv     OK   400       21     0.00
        bureau            CIBIL_RAW_PULL.csv     OK   400       22     0.00
bureau_refresh         BUREAU_REFRESH_6M.csv     OK  1600       16     1.56
bank_statement    BANK_STATEMENT_SUMMARY.csv     OK   400       23     0.00
        income    INCOME_DOCUMENT_PARSED.csv     OK   400       15     3.87
   dpd_history       DPD_HISTORY_MONTHLY.csv     OK 10800        9     0.00
     portfolio ACCOUNT_PORTFOLIO_MONTHLY.csv     OK  2400       10     9.41
     repayment    REPAYMENT_TRANSACTIONS.csv     OK  4578       12    14.76
    settlement          SETTLEMENT_CASES.csv     OK    30       11     0.00
      writeoff            WRITE_OFF_DATA.csv     OK    40      

In [5]:
# ============================================================
# CELL 2 — validate_schema.py
# Checks required columns exist per table
# Uses  →  dfs  (from Cell 1)
# ============================================================

SCHEMAS = {
    "customer": [
        "customer_id", "age", "gender", "occupation_type",
        "annual_income", "kyc_status",
    ],
    "loan": [
        "loan_id", "customer_id", "product_code", "sanction_amount",
        "interest_rate", "tenure_months", "loan_status", "disbursement_date",
    ],
    "bureau": [
        "customer_id", "bureau_score", "max_dpd_12m", "max_dpd_24m",
        "credit_utilisation_pct", "recent_enquiries_3m", "number_of_defaults",
    ],
    "application": [
        "customer_id", "declared_income", "foir", "ltv_ratio",
        "work_experience_years", "co_applicant_flag",
    ],
    "bank_statement": [
        "customer_id", "avg_monthly_credit", "avg_eod_balance",
        "emi_bounce_count", "salary_flag", "cash_withdrawal_pct",
    ],
    "dpd_history": [
        "loan_id", "month_end_date", "dpd", "overdue_amount",
        "bounce_flag", "mob",
    ],
    "income": [
        "customer_id", "declared_income", "verified_income", "mismatch_flag",
    ],
}

validation_results = []

print("=" * 60)
print("  STEP 2 — SCHEMA VALIDATION")
print("=" * 60)

schema_ok = True

for table, required in SCHEMAS.items():
    if table not in dfs:
        validation_results.append({
            "table":   table,
            "result":  "SKIPPED",
            "missing": "— table not loaded",
        })
        continue

    missing = sorted(set(required) - set(dfs[table].columns))
    if missing:
        schema_ok = False
        validation_results.append({
            "table":   table,
            "result":  "FAIL ✗",
            "missing": ", ".join(missing),
        })
    else:
        validation_results.append({
            "table":   table,
            "result":  "PASS ✓",
            "missing": "—",
        })

validation_df = pd.DataFrame(validation_results)
print(validation_df.to_string(index=False))
print(f"\n  Overall : {'ALL SCHEMAS VALID ✓' if schema_ok else 'ERRORS FOUND ✗'}")
print("=" * 60)


  STEP 2 — SCHEMA VALIDATION
         table result missing
      customer PASS ✓       —
          loan PASS ✓       —
        bureau PASS ✓       —
   application PASS ✓       —
bank_statement PASS ✓       —
   dpd_history PASS ✓       —
        income PASS ✓       —

  Overall : ALL SCHEMAS VALID ✓


In [7]:
# ============================================================
# CELL 3 — bureau_features.py
# CIBIL score, DPD flags, utilisation, enquiries
# Uses  →  dfs["bureau"]  (from Cell 1)
# Produces  →  bureau_feats
# ============================================================

_df = dfs["bureau"].copy()

print("=" * 60)
print("  STEP 3 — BUREAU FEATURES")
print(f"  Input  : {_df.shape[0]:,} rows × {_df.shape[1]} cols")
print("=" * 60)

# ── Binary flags ──────────────────────────────────────────────
_df["suit_filed_flag"]     = _df["suit_filed_flag"].fillna(0).astype(int)
_df["wilful_default_flag"] = _df["wilful_default_flag"].fillna(0).astype(int)
_df["ever_dpd90_12m"]      = (_df["max_dpd_12m"] >= 90).astype(int)
_df["ever_dpd90_24m"]      = (_df["max_dpd_24m"] >= 90).astype(int)
_df["has_writeoff"]        = (_df["written_off_accounts"] > 0).astype(int)
_df["high_enquiry_flag"]   = (_df["recent_enquiries_3m"] >= 3).astype(int)
_df["long_vintage_flag"]   = (_df["oldest_account_vintage_months"] >= 60).astype(int)

# ── Ratio features ────────────────────────────────────────────
_df["unsecured_mix_ratio"] = np.where(
    _df["total_tradelines"] > 0,
    _df["unsecured_tradelines"] / _df["total_tradelines"],
    0,
)
_df["enquiry_per_tradeline"] = np.where(
    _df["total_tradelines"] > 0,
    _df["recent_enquiries_6m"] / _df["total_tradelines"],
    _df["recent_enquiries_6m"],
)

# ── Bureau score band (1 = worst → 5 = best) ─────────────────
_df["bureau_score_band_num"] = pd.cut(
    _df["bureau_score"],
    bins=[0, 599, 649, 699, 749, 900],
    labels=[1, 2, 3, 4, 5],
).astype(float)

# ── Impute numerics (median) / categoricals (mode) ───────────
for col in _df.select_dtypes(include=[np.number]).columns:
    if _df[col].isnull().any():
        _df[col] = _df[col].fillna(_df[col].median())
for col in _df.select_dtypes(include=["object", "category"]).columns:
    if _df[col].isnull().any():
        _df[col] = _df[col].fillna(_df[col].mode().iloc[0])

# ── Winsorise key columns (1st–99th pct) ─────────────────────
for col in ["credit_utilisation_pct", "total_outstanding", "recent_enquiries_6m"]:
    if col in _df.columns:
        lo, hi = _df[col].quantile(0.01), _df[col].quantile(0.99)
        _df[col] = _df[col].clip(lo, hi)

bureau_feats = _df
del _df

# ── Output ────────────────────────────────────────────────────
_show = [
    "customer_id", "bureau_score", "bureau_score_band_num",
    "credit_utilisation_pct", "ever_dpd90_12m", "ever_dpd90_24m",
    "has_writeoff", "high_enquiry_flag", "long_vintage_flag",
    "unsecured_mix_ratio", "enquiry_per_tradeline",
    "suit_filed_flag", "wilful_default_flag",
]
print(f"\n  Output : {bureau_feats.shape[0]:,} rows × {bureau_feats.shape[1]} cols")
print(f"  New features added : {bureau_feats.shape[1] - dfs['bureau'].shape[1]}\n")
print(bureau_feats[[c for c in _show if c in bureau_feats.columns]].head(10).to_string(index=False))
print("=" * 60)


  STEP 3 — BUREAU FEATURES
  Input  : 400 rows × 22 cols

  Output : 400 rows × 30 cols
  New features added : 8

 customer_id  bureau_score  bureau_score_band_num  credit_utilisation_pct  ever_dpd90_12m  ever_dpd90_24m  has_writeoff  high_enquiry_flag  long_vintage_flag  unsecured_mix_ratio  enquiry_per_tradeline  suit_filed_flag  wilful_default_flag
     2000000           783                    5.0                   96.96               0               0             0                  1                  1             4.000000               1.000000                0                    0
     2000001           641                    2.0                   96.12               0               1             0                  1                  0             0.285714               0.428571                0                    0
     2000002           885                    5.0                   74.12               0               1             0                  1                  0         

In [9]:
# ============================================================
# CELL 4 — behavioral_features.py
# PAR30/60/90, MoB bands, bounce rate, NPA flag
# Uses  →  dfs["dpd_history"]  (from Cell 1)
# Produces  →  behav_feats
# ============================================================

_d = dfs["dpd_history"].copy()
_d["bounce_flag"] = _d["bounce_flag"].astype(int)

print("=" * 60)
print("  STEP 4 — BEHAVIORAL FEATURES")
print(f"  Input  : {len(_d):,} rows × {_d.shape[1]} cols")
print(f"  Unique loans : {_d['loan_id'].nunique():,}")
print("=" * 60)

# ── Core aggregations ─────────────────────────────────────────
_agg = _d.groupby("loan_id").agg(
    max_dpd_ever         =("dpd", "max"),
    max_overdue_amount   =("overdue_amount", "max"),
    total_overdue_amount =("overdue_amount", "sum"),
    total_bounces        =("bounce_flag", "sum"),
    total_mob            =("mob", "count"),
    max_mob              =("mob", "max"),
).reset_index()

_agg["bounce_rate"] = np.where(
    _agg["total_mob"] > 0,
    _agg["total_bounces"] / _agg["total_mob"],
    0,
)

# ── PAR flags: max DPD in rolling MoB windows ─────────────────
_r12 = (_d[_d["mob"] <= 12].groupby("loan_id")["dpd"]
        .max().rename("max_dpd_12m_mob").reset_index())
_r6  = (_d[_d["mob"] <= 6 ].groupby("loan_id")["dpd"]
        .max().rename("max_dpd_6m_mob").reset_index())

# ── Last delinquency MoB ──────────────────────────────────────
_last = (_d[_d["dpd"] > 0].groupby("loan_id")["mob"]
         .max().rename("last_delinq_mob").reset_index())

# ── Ever-NPA flag (fixed: single reset_index with name=) ──────
_npa = (_d[_d["asset_class"].isin(["NPA", "D1", "D2", "D3", "LOSS"])]
        .groupby("loan_id")
        .size()
        .gt(0)
        .astype(int)
        .reset_index(name="ever_npa_flag"))

# ── Merge laterals ────────────────────────────────────────────
_agg = (_agg
        .merge(_r12,  on="loan_id", how="left")
        .merge(_r6,   on="loan_id", how="left")
        .merge(_last, on="loan_id", how="left")
        .merge(_npa,  on="loan_id", how="left"))

_agg["months_since_last_delinquency"] = _agg["max_mob"] - _agg["last_delinq_mob"]

_agg.fillna({
    "max_dpd_12m_mob":               0,
    "max_dpd_6m_mob":                0,
    "months_since_last_delinquency": 999,
    "ever_npa_flag":                 0,
    "last_delinq_mob":               999,
}, inplace=True)

# ── Derived stress flags ──────────────────────────────────────
_agg["chronic_delinquent"] = (_agg["bounce_rate"] >= 0.25).astype(int)
_agg["recent_stress_flag"] = (_agg["max_dpd_6m_mob"] >= 30).astype(int)

behav_feats = _agg
del _d, _agg, _r12, _r6, _last, _npa

# ── Output ────────────────────────────────────────────────────
_show = [
    "loan_id", "max_dpd_ever", "bounce_rate",
    "max_dpd_12m_mob", "max_dpd_6m_mob",
    "months_since_last_delinquency", "ever_npa_flag",
    "chronic_delinquent", "recent_stress_flag",
]
print(f"\n  Output : {behav_feats.shape[0]:,} rows × {behav_feats.shape[1]} cols\n")
print(behav_feats[[c for c in _show if c in behav_feats.columns]].head(10).to_string(index=False))
print("=" * 60)


  STEP 4 — BEHAVIORAL FEATURES
  Input  : 10,800 rows × 9 cols
  Unique loans : 300

  Output : 300 rows × 15 cols

 loan_id  max_dpd_ever  bounce_rate  max_dpd_12m_mob  max_dpd_6m_mob  months_since_last_delinquency  ever_npa_flag  chronic_delinquent  recent_stress_flag
 1000001            92     0.472222               61              31                              1            0.0                   1                   1
 1000002            33     0.305556               32              31                              5            0.0                   1                   1
 1000005           120     0.638889              120              91                              0            0.0                   1                   1
 1000006           152     0.527778               31              31                              0            0.0                   1                   1
 1000007           153     0.583333               64              61                              0          

In [11]:
# ============================================================
# CELL 5 — application_features.py
# FOIR, LTV, income mismatch, co-applicant, employment risk
# Uses  →  dfs["application"], dfs["income"]  (from Cell 1)
# Produces  →  app_feats
# ============================================================

EMP_RISK = {
    "Salaried - Government":        1,
    "Salaried - PSU":               1,
    "Salaried - MNC":               2,
    "Salaried - Private":           2,
    "Self Employed - Professional": 3,
    "Business - Proprietorship":    3,
    "Business - Partnership":       3,
    "Business - Private Limited":   3,
    "Business - LLP":               3,
    "Contract Employee":            4,
    "Freelancer":                   5,
}

_d = dfs["application"].copy()

print("=" * 60)
print("  STEP 5 — APPLICATION FEATURES")
print(f"  Application input : {_d.shape[0]:,} rows × {_d.shape[1]} cols")
print(f"  Income input      : {dfs['income'].shape[0]:,} rows × {dfs['income'].shape[1]} cols")
print("=" * 60)

# ── Merge income verification ─────────────────────────────────
_income_cols = [
    "customer_id", "income_deviation_pct", "mismatch_flag",
    "verified_income", "itr_assessed_income", "verification_status",
]
_income_cols = [c for c in _income_cols if c in dfs["income"].columns]
_d = _d.merge(dfs["income"][_income_cols], on="customer_id", how="left")

# ── Employment risk rank ──────────────────────────────────────
_d["employment_risk_rank"] = _d["employment_type"].map(EMP_RISK).fillna(4)

# ── Binary flags ──────────────────────────────────────────────
_d["mismatch_flag"]     = _d["mismatch_flag"].fillna(0).astype(int)
_d["co_applicant_flag"] = _d["co_applicant_flag"].fillna(0).astype(int)
_d["high_foir_flag"]    = (_d["foir"] > 0.60).astype(int)
_d["high_ltv_flag"]     = (_d["ltv_ratio"] > 0.85).astype(int)
_d["low_experience"]    = (_d["work_experience_years"] < 2).astype(int)

# ── Income ratio features ─────────────────────────────────────
_d["declared_to_verified_ratio"] = np.where(
    _d["verified_income"] > 0,
    _d["declared_income"] / _d["verified_income"],
    1.0,
)
_d["itr_to_declared_ratio"] = np.where(
    _d["declared_income"] > 0,
    _d["itr_assessed_income"].fillna(0) / _d["declared_income"],
    0,
)

# ── Effective income (co-applicant uplift) ────────────────────
_d["effective_income"] = np.where(
    _d["co_applicant_flag"] == 1,
    _d["declared_income"] + _d["co_applicant_income"].fillna(0),
    _d["declared_income"],
)

# ── Loan-to-income ratio ──────────────────────────────────────
_d["loan_to_income_ratio"] = np.where(
    _d["effective_income"] > 0,
    _d["requested_amount"].fillna(0) / _d["effective_income"],
    0,
)

app_feats = _d
del _d

# ── Output ────────────────────────────────────────────────────
_show = [
    "customer_id", "declared_income", "verified_income",
    "effective_income", "declared_to_verified_ratio",
    "itr_to_declared_ratio", "loan_to_income_ratio",
    "foir", "ltv_ratio", "employment_risk_rank",
    "mismatch_flag", "high_foir_flag", "high_ltv_flag",
    "co_applicant_flag", "low_experience",
]
print(f"\n  Output : {app_feats.shape[0]:,} rows × {app_feats.shape[1]} cols\n")
print(app_feats[[c for c in _show if c in app_feats.columns]].head(10).to_string(index=False))
print("=" * 60)


  STEP 5 — APPLICATION FEATURES
  Application input : 400 rows × 21 cols
  Income input      : 400 rows × 15 cols

  Output : 400 rows × 34 cols

 customer_id  declared_income  verified_income  effective_income  declared_to_verified_ratio  itr_to_declared_ratio  loan_to_income_ratio   foir  ltv_ratio  employment_risk_rank  mismatch_flag  high_foir_flag  high_ltv_flag  co_applicant_flag  low_experience
     2000000       7028210.08       7265220.89        7274107.94                    0.967377               0.798115              1.063827 0.6454     0.8964                   3.0              0               1              1                  1               0
     2000001       5960334.01       6360194.42        5960334.01                    0.937131               0.925075              1.597044 0.5863     0.7975                   3.0              0               0              0                  0               0
     2000002      11145999.29      12255687.84       11145999.29             

In [13]:
# ============================================================
# CELL 6 — create_features.py
# Bank features + joins + target + imputation → master dataset
# Uses  →  dfs, bureau_feats, behav_feats, app_feats
#          (all produced by Cells 1–5)
# Produces  →  master  (saved to data/processed/model_dataset.csv)
# ============================================================

OUTPUT_PATH = "data/processed/model_dataset.csv"

OUTLIER_COLS = [
    "bureau_score", "credit_utilisation_pct", "total_outstanding",
    "avg_monthly_credit", "avg_eod_balance", "declared_income",
    "sanction_amount", "max_overdue_amount", "loan_to_income_ratio",
]

print("=" * 60)
print("  STEP 6 — CREATE MASTER FEATURE DATASET")
print("=" * 60)

# ── Bank statement features ───────────────────────────────────
_bs = dfs["bank_statement"].copy()

for _col in ["salary_flag", "gaming_txn_flag", "casino_txn_flag", "international_txn_flag"]:
    if _col in _bs.columns:
        _bs[_col] = _bs[_col].astype(int)

_bs["net_cashflow_ratio"] = np.where(
    _bs["avg_monthly_credit"] > 0,
    (_bs["avg_monthly_credit"] - _bs["avg_monthly_debit"]) / _bs["avg_monthly_credit"],
    0,
)
_bs["emi_to_income_ratio"] = np.where(
    _bs["avg_monthly_credit"] > 0,
    _bs["avg_monthly_emi_outflow"] / _bs["avg_monthly_credit"],
    0,
)
_bs["balance_to_income_ratio"] = np.where(
    _bs["avg_monthly_credit"] > 0,
    _bs["avg_eod_balance"] / _bs["avg_monthly_credit"],
    0,
)

_total_bounces = (
    _bs["emi_bounce_count"]
    + _bs["inward_cheque_bounce_count"]
    + _bs["outward_cheque_bounce_count"]
)
_bs["high_stress_flag"] = (_total_bounces >= 3).astype(int)
_bs["high_cash_flag"]   = (_bs["cash_withdrawal_pct"] > 40).astype(int)

bank_feats = _bs
del _bs, _total_bounces
print(f"  Bank features  : {bank_feats.shape[0]:,} rows × {bank_feats.shape[1]} cols")

# ── Target variable ───────────────────────────────────────────
# Bad = DPD >= 90 in MoB 7–18  OR  settlement  OR  write-off
_dpd = dfs["dpd_history"]
_perf = _dpd[(_dpd["mob"] >= 7) & (_dpd["mob"] <= 18)]
_target = (_perf
           .groupby("loan_id")
           .agg(target_default=("dpd", lambda x: int((x >= 90).any())))
           .reset_index())

_settled_ids  = set(dfs["settlement"]["loan_id"]) if "settlement" in dfs and len(dfs["settlement"]) else set()
_writeoff_ids = set(dfs["writeoff"]["loan_id"])   if "writeoff"   in dfs and len(dfs["writeoff"])   else set()

_target["target_default"] = _target.apply(
    lambda r: 1 if (
        r["target_default"] == 1
        or r["loan_id"] in _settled_ids
        or r["loan_id"] in _writeoff_ids
    ) else 0,
    axis=1,
)
del _dpd, _perf, _settled_ids, _writeoff_ids
print(f"  Target built   : Default rate = {_target['target_default'].mean()*100:.2f}%  |  N = {len(_target):,}")

# ── Loan spine ────────────────────────────────────────────────
_loan_cols = [
    "loan_id", "customer_id", "product_code",
    "sanction_amount", "interest_rate", "tenure_months", "sourcing_channel",
]
_loan_cols = [c for c in _loan_cols if c in dfs["loan"].columns]
_loan = dfs["loan"][_loan_cols].copy()

# ── Join everything ───────────────────────────────────────────
master = (
    _loan
    .merge(_target,      on="loan_id",     how="inner")
    .merge(behav_feats,  on="loan_id",     how="left")
    .merge(bureau_feats, on="customer_id", how="left")
    .merge(bank_feats,   on="customer_id", how="left")
    .merge(app_feats,    on="customer_id", how="left", suffixes=("", "_app"))
)
del _loan, _target
print(f"  After join     : {master.shape[0]:,} rows × {master.shape[1]} cols")

# ── Impute (median for numeric, mode for categorical) ─────────
for _col in master.select_dtypes(include=[np.number]).columns:
    if master[_col].isnull().any():
        master[_col] = master[_col].fillna(master[_col].median())
for _col in master.select_dtypes(include=["object", "category"]).columns:
    if master[_col].isnull().any():
        master[_col] = master[_col].fillna(master[_col].mode().iloc[0])

# ── Winsorise (1st–99th percentile) ──────────────────────────
for _col in OUTLIER_COLS:
    if _col in master.columns:
        _lo, _hi = master[_col].quantile(0.01), master[_col].quantile(0.99)
        master[_col] = master[_col].clip(_lo, _hi)

# ── Save ──────────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
master.to_csv(OUTPUT_PATH, index=False)
print(f"  Saved          → {OUTPUT_PATH}")

# ── Output ────────────────────────────────────────────────────
_show = [
    "loan_id", "customer_id", "product_code", "sanction_amount",
    "tenure_months", "target_default",
    "bureau_score", "bureau_score_band_num",
    "max_dpd_ever", "bounce_rate", "ever_npa_flag",
    "loan_to_income_ratio", "foir", "high_foir_flag",
    "net_cashflow_ratio", "high_stress_flag",
]

print(f"\n  FINAL SHAPE : {master.shape[0]:,} rows × {master.shape[1]} cols")
print("\n  Target distribution:")
print(
    master["target_default"]
    .value_counts(normalize=True)
    .rename(index={0: "Good (0)", 1: "Bad (1)"})
    .apply(lambda x: f"{x:.2%}")
    .to_string()
)

_null_check = (master[[c for c in _show if c in master.columns]]
               .isnull().mean() * 100).round(2)
_has_nulls = _null_check[_null_check > 0]
print("\n  Null % in key columns :",
      "None — fully imputed ✓" if _has_nulls.empty else "")
if not _has_nulls.empty:
    print(_has_nulls.to_string())

print(f"\n  Sample output (key features):")
print("-" * 60)
print(master[[c for c in _show if c in master.columns]].head(10).to_string(index=False))
print("=" * 60)


  STEP 6 — CREATE MASTER FEATURE DATASET
  Bank features  : 400 rows × 28 cols
  Target built   : Default rate = 51.67%  |  N = 300
  After join     : 300 rows × 111 cols
  Saved          → data/processed/model_dataset.csv

  FINAL SHAPE : 300 rows × 111 cols

  Target distribution:
target_default
Bad (1)     51.67%
Good (0)    48.33%

  Null % in key columns : None — fully imputed ✓

  Sample output (key features):
------------------------------------------------------------
 loan_id  customer_id product_code  sanction_amount  tenure_months  target_default  bureau_score  bureau_score_band_num  max_dpd_ever  bounce_rate  ever_npa_flag  loan_to_income_ratio   foir  high_foir_flag  net_cashflow_ratio  high_stress_flag
 1000001      2000001           HL       4775392.47             12               1         641.0                    2.0            92     0.472222            0.0              1.597044 0.5863               0            0.518233                 1
 1000002      2000002        